In [201]:
import pandas as pd
import os
from datetime import datetime, timedelta

%pip install python-dateutil

from dateutil import parser
import numpy as np

Note: you may need to restart the kernel to use updated packages.


## Compile Dataset Methods

In [202]:
date_time_format = "%m-%d-%Y %H:%M"
compiled_dates = {}

In [203]:
"""
Check to see if inputted dataset has more than 14 days by parsing data/time column in files

    Input: 
        orgDir - path of the directory where the original csv files are
        fileName - name of file being observed
    Return: fileName param, boolean, List
    boolean representing if number of days greater than 14 days and list of days that
    need to be removed
"""
def isFourteenDays(filePath, fileName, newDirectory):
    # Load data
    df = pd.read_csv(filePath)

    # Parse the date column (assuming it's named 'Date'). Adjust if the name is different.
    times = pd.to_datetime(df['time'])
    
    # Calculate date range
    min_date = times.min()
    max_date = times.max()
    days_difference = (max_date - min_date).days
    
    # Identify extra dates beyond the 14-day range
    if days_difference > 14:
        cutoff_date = min_date + timedelta(days=14)
        extra_days = df[times > cutoff_date]['time'].unique().tolist()
        return filePath, True, extra_days
    else:
        new_file_path = os.path.join(newDirectory, f"{os.path.basename(fileName)}")
        for i in range(len(df['time'])):
            curr_time = df.loc[i, "time"]
            try:
                # Check if the string is already in the desired format
                if datetime.strptime(curr_time, date_time_format):
                    continue  # It's already in the correct format
            except ValueError:
                # Parse and reformat the time if not in the correct format
                try:
                    parsed_time = parser.parse(curr_time)
                    df.loc[i, "time"] = parsed_time.strftime(date_time_format)
                except Exception as e:
                    return f"Error: {curr_time} - {e}"
        df.to_csv(new_file_path, index=False)
        return filePath, False, []

"""
Using the method above, create a new file that will have the extra days removed from the set
    Input: orgDir, fileName, removeDays, newdirectory
        fileName - name of file beign observed
        removeDays - List of days that need to be removed
        newdirectory - the directory path where the new file will be stored
    Return: None
"""
def removeExtraDays(filePath, fileName, removeDays, newdirectory):
    # Load the data
    df = pd.read_csv(filePath)
    
    # Convert date column to datetime format for comparison
    times = pd.to_datetime(df['time'])
    
    # Remove the specified days from the dataset
    df_filtered = df[~times.isin(pd.to_datetime(removeDays))]
    
    # Ensure the output directory exists
    if not os.path.exists(newdirectory):
        os.makedirs(newdirectory)
    
    # Save the filtered file in the new directory
    new_file_path = os.path.join(newdirectory, f"{os.path.basename(fileName)}")
    for i in range(len(df_filtered['time'])):
        curr_time = df_filtered.loc[i, "time"]
        try:
            # Check if the string is already in the desired format
            if datetime.strptime(curr_time, date_time_format):
                continue  # It's already in the correct format
        except ValueError:
            # Parse and reformat the time if not in the correct format
            try:
                parsed_time = parser.parse(curr_time)
                df_filtered.loc[i, "time"] = parsed_time.strftime(date_time_format)
            except Exception as e:
                return f"Error: {curr_time} - {e}"
    df_filtered.to_csv(new_file_path, index=False)

"""
Using the files in the directory, concat files into a new dataset. Save the new dataset inside
of the new directory
Make sure to add a new column that represents the patient ID before concatting files
    Input: oldDirectory, newDirectory
        oldDirectory - the directory path where the preprocessing dats is compiled
        newDirectory - the directory path where the concatted data will be stored
    oldDirectory and newDirectory are not going to be same so files will be easily differentiable
    Return: None
"""
##This method needs to be debugged. We may not need to oldDirectory parameter if we add the files
## that have at most 14 days into the new directory in isFouteenDays possibly
def concatFiles(oldDirectory, newDirectory):
    all_data = []
    patient_id = 1
    
    # Loop through each file in the old directory
    for file in os.listdir(oldDirectory):
        if file.endswith(".csv"):
            file_path = os.path.join(oldDirectory, file)
            
            # Load file data
            df = pd.read_csv(file_path)
            
            # Add PatientID column
            df['PatientID'] = patient_id
            all_data.append(df)
            
            # Increment PatientID for the next file
            patient_id += 1
    
    # Concatenate all dataframes in the list
    concatenated_df = pd.concat(all_data, ignore_index=True)

    firstcol = concatenated_df.pop('PatientID')
    concatenated_df.insert(0, 'PatientID', firstcol)

    # Ensure the output directory exists
    if not os.path.exists(newDirectory):
        os.makedirs(newDirectory)
    
    # Save the concatenated dataframe
    output_file = os.path.join(newDirectory, "concatenated_data.csv")
    concatenated_df.to_csv(output_file, index=False)

In [204]:
##Find List of Dates to find in the Raw Data Folder for the associated Patient Id
def getListOfDays(filePath):
    df = pd.read_csv(filePath)
    all_steps = df['time']
    all_steps = all_steps.to_numpy()
    dates = []
    for curr_step in all_steps:
        date_time_obj = datetime.strptime(curr_step, date_time_format)
        curr_date = date_time_obj.date()
        curr_date = curr_date.strftime("%Y-%m-%d")
        if curr_date not in dates:
            dates.append(curr_date)
    return dates

In [205]:
def delete_files_in_directory(directory_path):
   try:
     files = os.listdir(directory_path)
     for file in files:
       file_path = os.path.join(directory_path, file)
       if os.path.isfile(file_path):
         os.remove(file_path)
     print("All files deleted successfully.")
   except OSError:
     print("Error occurred while deleting files.")

## TESTING

In [206]:
##Getting original directory from preprocessing data
orgDataDir = os.path.join(os.getcwd(), "HUPA-UCM Diabetes Dataset")
orgDataDir = os.path.join(orgDataDir, 'Preprocessed')

print(orgDataDir)

##Getting the list of files within the directory
fileList = os.listdir(orgDataDir)
print(fileList)

##Getting the path of the directory where the new files will be stored
concatDataDir = os.path.join(os.getcwd(), "Final_Dataset")
filteredDataDir = os.path.join(concatDataDir, "Filtered_Data")
print(concatDataDir)
print(filteredDataDir)

c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\HUPA-UCM Diabetes Dataset\Preprocessed
['HUPA0001P.csv', 'HUPA0002P.csv', 'HUPA0003P.csv', 'HUPA0004P.csv', 'HUPA0005P.csv', 'HUPA0006P.csv', 'HUPA0007P.csv', 'HUPA0009P.csv', 'HUPA0010P.csv', 'HUPA0011P.csv', 'HUPA0014P.csv', 'HUPA0015P.csv', 'HUPA0016P.csv', 'HUPA0017P.csv', 'HUPA0018P.csv', 'HUPA0019P.csv', 'HUPA0020P.csv', 'HUPA0021P.csv', 'HUPA0022P.csv', 'HUPA0023P.csv', 'HUPA0024P.csv', 'HUPA0025P.csv', 'HUPA0026P.csv', 'HUPA0027P.csv', 'HUPA0028P.csv']
c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\Final_Dataset
c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\Final_Dataset\Filtered_Data


#### Check files that have less than or equal to 14 dates

##### Need to test isFourteenDays()

In [207]:
# # ##Get the path of the file that needs to be opened
# file1_name = "HUPA0010P.csv"
# file_path1 = os.path.join(orgDataDir, file1_name)
# # ##Check to see whether this is 14 days worth of data
# fileName, isfourteen, extraDaysList = isFourteenDays(file_path1, file1_name, filteredDataDir)
# print("File Observed", fileName)
# print("Does Dataset Contain Fourteen Days: ", isfourteen)
# print("Extra Days", extraDaysList)

# getListOfDays(file_path1)

#### Check files that have greater than 14 dates
##### Test the functions isFourteenDays() and removeExtraDays()

In [208]:
# file2_name = 'HUPA0027P.csv'
# file_path2 = os.path.join(orgDataDir, file2_name)
# fileName, isfourteen, extraDaysList = isFourteenDays(file_path2, file2_name, filteredDataDir)
# print("##Check if 14 Days##")
# print("File Observed", fileName)
# print("Does Dataset Contain Fourteen Days: ", isfourteen)
# print("Extra Days", extraDaysList, "\n")


# print("##Remove Days Test##")
# removeExtraDays(file_path2, file2_name, extraDaysList, filteredDataDir)

# filteredData2 = os.path.join(filteredDataDir, file2_name)
# fileName, isfourteen, extraDaysList = isFourteenDays(filteredData2, file2_name, filteredDataDir)
# print("##Check if New File is 14 Days##")
# print("File Observed", fileName)
# print("Does Dataset Contain Fourteen Days: ", isfourteen)
# print("Extra Days", extraDaysList, "\n")

#### Check if concatenation works properly

In [209]:
# concatFiles(filteredDataDir, concatDataDir)

In [210]:
delete_files_in_directory(filteredDataDir)
# delete_files_in_directory(concatDataDir)

All files deleted successfully.


## ADD SLEEP FEATURE TO ORIGINAL DATASET

In [211]:
##Starting Point of Path to get to sleep data
sleep_directory = os.path.join(os.getcwd(), "HUPA-UCM Diabetes Dataset")
sleep_directory = os.path.join(sleep_directory, 'Raw_Data')
##List of Folders for Each Patient ID
sleep_dir_folders = os.listdir(sleep_directory)

## Dictionary that stores the patient ID as the key and the value is the path to the files for 
## the night summary and the night tracking
night_files = {}
for dir in sleep_dir_folders:
    sleep_path = os.path.join(sleep_directory, dir, "fitbit")
    temp_list = []
    for file in os.listdir(sleep_path):
        if "_night" in file:
            temp_list.append(os.path.join(sleep_path, file))
    night_files[dir] = temp_list

In [212]:
##return number of wake states from the night files
def calculateAwakening(fileName):
    wake_state_count = 0
    data = pd.read_csv(fileName)
    for i in range(len(data['State'])):
        if (data['State'][i] == "wake"):
            wake_state_count += 1
    return wake_state_count

##return the efficiency and the minutes awake from the night summary file (WASO = Minutes Awake Vals)
def efficiencyandWASOCalc(fileName):
    data = pd.read_csv(fileName)
    # print(data)
    # print(data['Efficiency'])
    # print(data['Minutes Awake'])
    return int(data['Efficiency'].iloc[0]), int(data['Minutes Awake'].iloc[0])

##Finding the approximate time step to add the sleep data to and getting the date as one of the keys for the dictionary for
##data checks
def getNextTimeStep(filePath):
    data = pd.read_csv(filePath)
    endtime = data.iloc[0]['End Time']
    next_time_step = datetime.strptime(endtime, "%Y-%m-%dT%H:%M:%S.000")
    
    minute = next_time_step.minute
    if minute % 5 != 0:
        next_minute = (minute // 5 + 1) * 5
        if next_minute == 60:
            next_time_step =next_time_step.replace(minute=0, second=0) + timedelta(hours=1)
        else:
            next_time_step =next_time_step.replace(minute=next_minute, second=0)

    time_step_str = next_time_step.strftime(date_time_format)
    return time_step_str

## TESTING

In [ ]:
file = night_files["HUPA0001P"][0]
print(file)
awake_state_count = calculateAwakening(file)
file1 = night_files["HUPA0001P"][1]
print(file1)
efficiency, waso = efficiencyandWASOCalc(file1)
print(awake_state_count)
print(efficiency, waso)

approx_next_step = getNextTimeStep(file1)

print(approx_next_step)
print(type(approx_next_step))


c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\HUPA-UCM Diabetes Dataset\Raw_Data\HUPA0001P\fitbit\HUPA0001P_sleep_2018-06-14_night.csv
c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\HUPA-UCM Diabetes Dataset\Raw_Data\HUPA0001P\fitbit\HUPA0001P_sleep_2018-06-14_night_summary.csv
7
94 46
06-14-2018 06-14-2018 10:10
<class 'str'>


### Call Methods to Filter Out the Data

In [215]:
print(os.listdir(orgDataDir))
for file in os.listdir(orgDataDir):
    file_path = os.path.join(orgDataDir, file)
    fileName, isFourteen, extraDaysList = isFourteenDays(file_path, file, filteredDataDir)
    if isFourteen == True:
        removeExtraDays(file_path, file, extraDaysList, filteredDataDir)
    filteredPath = os.path.join(filteredDataDir, file)
    dates = getListOfDays(filteredPath)
    compiled_dates[file.replace(".csv", "")] = dates

['HUPA0001P.csv', 'HUPA0002P.csv', 'HUPA0003P.csv', 'HUPA0004P.csv', 'HUPA0005P.csv', 'HUPA0006P.csv', 'HUPA0007P.csv', 'HUPA0009P.csv', 'HUPA0010P.csv', 'HUPA0011P.csv', 'HUPA0014P.csv', 'HUPA0015P.csv', 'HUPA0016P.csv', 'HUPA0017P.csv', 'HUPA0018P.csv', 'HUPA0019P.csv', 'HUPA0020P.csv', 'HUPA0021P.csv', 'HUPA0022P.csv', 'HUPA0023P.csv', 'HUPA0024P.csv', 'HUPA0025P.csv', 'HUPA0026P.csv', 'HUPA0027P.csv', 'HUPA0028P.csv']


In [ ]:
compiled_dates

{'HUPA0001P': ['2018-06-13',
  '2018-06-14',
  '2018-06-15',
  '2018-06-16',
  '2018-06-17',
  '2018-06-18',
  '2018-06-19',
  '2018-06-20',
  '2018-06-21',
  '2018-06-22',
  '2018-06-23',
  '2018-06-24',
  '2018-06-25',
  '2018-06-26',
  '2018-06-27'],
 'HUPA0002P': ['2018-06-13',
  '2018-06-14',
  '2018-06-15',
  '2018-06-16',
  '2018-06-17',
  '2018-06-18',
  '2018-06-19',
  '2018-06-20',
  '2018-06-21',
  '2018-06-22',
  '2018-06-23',
  '2018-06-24'],
 'HUPA0003P': ['2018-06-13',
  '2018-06-14',
  '2018-06-15',
  '2018-06-16',
  '2018-06-17',
  '2018-06-18',
  '2018-06-19',
  '2018-06-20',
  '2018-06-21',
  '2018-06-22',
  '2018-06-23',
  '2018-06-24',
  '2018-06-25',
  '2018-06-26'],
 'HUPA0004P': ['2018-07-09',
  '2018-07-10',
  '2018-07-11',
  '2018-07-12',
  '2018-07-13',
  '2018-07-14',
  '2018-07-15',
  '2018-07-16',
  '2018-07-17',
  '2018-07-18',
  '2018-07-19',
  '2018-07-20'],
 'HUPA0005P': ['2018-07-09',
  '2018-07-10',
  '2018-07-11',
  '2018-07-12',
  '2018-07-13',
  '

### CALLING ALL METHODS ABOVE to create POOR or GOOD Sleep Recordings using night_files

##### Poor Sleep Quality = -1, No Sleep Quality = 0, Good Sleep Quality

In [228]:
print(filteredDataDir)

for file in os.listdir(filteredDataDir):
    print(file)
    file_path = os.path.join(filteredDataDir, file)
    patientID = file.replace(".csv", "")
    file_list, date_list = night_files[patientID], np.array(compiled_dates[patientID]).reshape(1, -1)
    print(date_list.shape)
    ##Set default values for awake_state, WASO and sleep efficiency to determine sleep quality
    awake_state_count, efficiency, waso = 0, 0, 0
    print(date_list)

    ##Add the default values for the sleep quality
    df = pd.read_csv(file_path)
    sleep_quality = np.zeros((len(df['time']), 1))
    df['sleep_quality'] = sleep_quality

    ##Shift the basal_rate feature to the end to represent the output of the model
    num_columns = df.shape[1]
    col = df.pop('basal_rate')
    df.insert(num_columns - 1, col.name, col)

    for night_file_path in file_list:
        # print(night_file_path, len(night_file_path))
        # print("Night = ", night_file_path[-20:-10])
        # print("Night Summary = ", night_file_path[-28:-18])
        if "night." in night_file_path and night_file_path[-20:-10] in date_list:
            awake_state_count = calculateAwakening(night_file_path)
        elif "night_summary" in night_file_path and night_file_path[-28:-18] in date_list:
            efficiency, waso = efficiencyandWASOCalc(night_file_path)
            approx_next_step = getNextTimeStep(night_file_path)
            # print(approx_next_step)

            print(efficiency, waso, awake_state_count)
            row_index = 0
            if approx_next_step in df['time'].values:
                row_index = df.loc[df['time'] == approx_next_step].index[0]
            else:
                continue

            # print(row_index)
            # print(awake_state_count, efficiency, waso)
            # print(df.loc[row_index, 'sleep_quality'])

            #Checks if efficiency and waso conditions are met
            if (efficiency < 85) and (waso > 40):
                df.loc[row_index, 'sleep_quality'] = -1
            else:
                ##If above condition is false, then check efficiency and awake_state to see if that combination creates poor sleep
                ##Also check to see if waso and awake_state is the combination that creates poor sleep
                if (efficiency < 85 and awake_state_count > 4) or (waso > 40 and awake_state_count > 4):
                    df.loc[row_index, 'sleep_quality'] = -1
                else: ##If neither of the above conditions are met, then the patient had good sleep
                    df.loc[row_index, 'sleep_quality'] = 1
            print(df.loc[row_index, 'sleep_quality'])
    df.to_csv(file_path, index=False)


c:\Users\nwalr\OneDrive - Georgia Institute of Technology\VIP\TreatmentSystems\Final_Dataset\Filtered_Data
HUPA0001P.csv
(1, 15)
[['2018-06-13' '2018-06-14' '2018-06-15' '2018-06-16' '2018-06-17'
  '2018-06-18' '2018-06-19' '2018-06-20' '2018-06-21' '2018-06-22'
  '2018-06-23' '2018-06-24' '2018-06-25' '2018-06-26' '2018-06-27']]
94 46 7
-1.0
90 62 4
1.0
98 35 4
1.0
92 31 2
1.0
95 49 4
1.0
94 49 4
1.0
95 28 3
1.0
91 73 6
-1.0
95 30 1
1.0
96 36 4
1.0
93 48 4
1.0
95 38 2
1.0
95 26 4
1.0
100 0 0
1.0
HUPA0002P.csv
(1, 12)
[['2018-06-13' '2018-06-14' '2018-06-15' '2018-06-16' '2018-06-17'
  '2018-06-18' '2018-06-19' '2018-06-20' '2018-06-21' '2018-06-22'
  '2018-06-23' '2018-06-24']]
98 24 3
1.0
97 31 1
1.0
94 51 4
1.0
97 39 5
1.0
97 26 3
1.0
99 32 3
1.0
97 31 3
1.0
100 20 1
1.0
98 22 1
1.0
99 23 1
1.0
96 68 5
-1.0
HUPA0003P.csv
(1, 14)
[['2018-06-13' '2018-06-14' '2018-06-15' '2018-06-16' '2018-06-17'
  '2018-06-18' '2018-06-19' '2018-06-20' '2018-06-21' '2018-06-22'
  '2018-06-23' '2018-0

## Concatenating All Filtered Data

In [ ]:
# concatFiles(filteredDataDir, concatDataDir)